# PanNuke Multi-Label Classification with DenseNet169
**Dataset:** PanNuke (Pan-Cancer Nuclei Instance Segmentation and Detection)  
**Task:** Multi-label classification — 5 tissue classes  
**Model:** DenseNet169 (pretrained ImageNet backbone)  
**Split:** Official PanNuke recommended split (Fold1+Fold2 → Train/Val, Fold3 → Test)

## 1. Imports & Environment Setup

In [ ]:
import os
import random
import json
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, multilabel_confusion_matrix,
    classification_report
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42

def set_seed(seed: int = SEED) -> None:
    """Fix all sources of randomness for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Configuration

In [ ]:
# ── Paths — adjust BASE_DIR to where your folds live ─────────────────────────
BASE_DIR   = Path('.')          # <- change to your dataset root if needed
OUTPUT_DIR = Path('./outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FOLD_DIRS: Dict[int, Dict[str, Path]] = {
    fold: {
        'images': BASE_DIR / f'fold_{fold}' / 'images' / f'fold_{fold}' / 'images.npy',
        'masks' : BASE_DIR / f'fold_{fold}' / 'masks'  / f'fold_{fold}' / 'masks.npy',
    }
    for fold in [1, 2, 3]
}

# ── Dataset constants ─────────────────────────────────────────────────────────
CLASS_NAMES: List[str] = [
    'Neoplastic',
    'Non-neoplastic Epithelial',
    'Inflammatory',
    'Connective',
    'Dead',
]
NUM_CLASSES = len(CLASS_NAMES)  # 5
# PanNuke masks shape: (N, H, W, 6)  — channels 0-4 = classes, channel 5 = background
CLASS_CHANNEL_INDICES = list(range(5))

# ── Training hyper-parameters ─────────────────────────────────────────────────
IMG_SIZE    = 256          # PanNuke native resolution
BATCH_SIZE  = 16           # safe for 8 GB VRAM; reduce to 8 if OOM
NUM_EPOCHS  = 30
LR          = 1e-4
WEIGHT_DECAY= 1e-4
NUM_WORKERS = 4
THRESHOLD   = 0.5         # sigmoid threshold for binary decisions
N_TEST_HOLD = 50          # images held-out as final test set

print('Configuration loaded.')
for fold, paths in FOLD_DIRS.items():
    exists = {k: v.exists() for k, v in paths.items()}
    print(f'  Fold {fold}: images={exists["images"]} | masks={exists["masks"]}')

## 3. Data Loading & Label Derivation

In [ ]:
def derive_multilabel_from_masks(masks: np.ndarray) -> np.ndarray:
    """
    Convert PanNuke pixel-wise masks → per-image multi-hot label vectors.

    masks : (N, H, W, 6)  — float32, channels 0-4 = nuclei classes, 5 = background
    returns: (N, 5)        — binary, 1 if class has ≥1 annotated pixel

    ⚠ Data-leakage note: label derivation is purely from pixel presence;
      no statistics from images are used, so no leakage risk here.
    """
    # Sum over spatial dims → (N, 6); take first 5 channels
    class_pixel_counts = masks[..., :5].sum(axis=(1, 2))   # (N, 5)
    labels = (class_pixel_counts > 0).astype(np.uint8)     # (N, 5)
    return labels


def load_fold(
    fold_id: int,
    mmap_mode: str = 'r',
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Memory-map a single fold's .npy files.
    mmap_mode='r' means we read slices on demand — crucial for 8 GB VRAM machines
    so we don't load all three folds into RAM at once.

    Returns
    -------
    images : (N, H, W, 3)  uint8
    labels : (N, 5)        uint8
    """
    paths = FOLD_DIRS[fold_id]
    images = np.load(paths['images'], mmap_mode=mmap_mode)   # lazy load
    masks  = np.load(paths['masks'],  mmap_mode=mmap_mode)   # lazy load

    print(f'  Fold {fold_id} — images: {images.shape} | masks: {masks.shape}')

    # Derive labels (efficient: mask summation, not full copy)
    labels = derive_multilabel_from_masks(np.array(masks))  # force into RAM for label ops
    return images, labels


print('Loading folds (memory-mapped)...')
fold1_images, fold1_labels = load_fold(1)
fold2_images, fold2_labels = load_fold(2)
fold3_images, fold3_labels = load_fold(3)

# Class distribution overview
all_labels_combined = np.vstack([fold1_labels, fold2_labels, fold3_labels])
print(f'\nTotal images  : {len(all_labels_combined)}')
print('Class presence (# images with at least 1 nucleus of each class):')
for i, name in enumerate(CLASS_NAMES):
    count = all_labels_combined[:, i].sum()
    print(f'  [{i}] {name:<30} {count:>5} ({100*count/len(all_labels_combined):.1f}%)')

## 4. Hold-Out 50 Test Images (stratified across all folds)

In [ ]:
def build_holdout_test_set(
    f1_imgs: np.ndarray, f1_lbl: np.ndarray,
    f2_imgs: np.ndarray, f2_lbl: np.ndarray,
    f3_imgs: np.ndarray, f3_lbl: np.ndarray,
    n_test: int = N_TEST_HOLD,
    seed: int = SEED,
) -> Tuple[
    np.ndarray, np.ndarray,           # test images + labels
    np.ndarray, np.ndarray,           # fold1 remainder
    np.ndarray, np.ndarray,           # fold2 remainder
    np.ndarray, np.ndarray,           # fold3 remainder
]:
    """
    Sample ~n_test/3 images from each fold so the hold-out set is
    representative of all folds.  Remainders go to train/val.
    """
    rng = np.random.default_rng(seed)
    per_fold = n_test // 3  # ≈16 from each; total will be 3*16=48, top up from fold3
    extra    = n_test - 3 * per_fold

    def sample_fold(imgs, lbls, n):
        idx = rng.choice(len(imgs), size=n, replace=False)
        mask = np.ones(len(imgs), dtype=bool)
        mask[idx] = False
        return imgs[idx], lbls[idx], imgs[mask], lbls[mask]

    t1_i, t1_l, r1_i, r1_l = sample_fold(f1_imgs, f1_lbl, per_fold)
    t2_i, t2_l, r2_i, r2_l = sample_fold(f2_imgs, f2_lbl, per_fold)
    t3_i, t3_l, r3_i, r3_l = sample_fold(f3_imgs, f3_lbl, per_fold + extra)

    test_imgs   = np.concatenate([t1_i, t2_i, t3_i], axis=0)
    test_labels = np.concatenate([t1_l, t2_l, t3_l], axis=0)

    print(f'Hold-out test set: {len(test_imgs)} images')
    return test_imgs, test_labels, r1_i, r1_l, r2_i, r2_l, r3_i, r3_l


(
    test_images, test_labels,
    r1_images, r1_labels,
    r2_images, r2_labels,
    r3_images, r3_labels,
) = build_holdout_test_set(
    fold1_images, fold1_labels,
    fold2_images, fold2_labels,
    fold3_images, fold3_labels,
)

print(f'  Remainder fold1 : {len(r1_images)}')
print(f'  Remainder fold2 : {len(r2_images)}')
print(f'  Remainder fold3 : {len(r3_images)}')

## 5. Official PanNuke Split  
**Recommended:** Fold1 + Fold2 remainders → Train (80%) / Val (20%) | Fold3 remainder → additional val/inference

In [ ]:
# Concatenate fold1+fold2 remainders as the training pool
# Fold3 remainder acts as additional validation (PanNuke paper recommendation)
trainval_images = np.concatenate([r1_images, r2_images], axis=0)
trainval_labels = np.concatenate([r1_labels, r2_labels], axis=0)
val_extra_images = r3_images
val_extra_labels = r3_labels

# Split fold1+fold2 pool into 80/20 train/val
# ⚠ Stratify on argmax (dominant class) since multilabel stratification
#   requires specialized libs; this is a reasonable proxy.
dominant_class = trainval_labels.argmax(axis=1)
train_idx, val_idx = train_test_split(
    np.arange(len(trainval_images)),
    test_size=0.20,
    stratify=dominant_class,
    random_state=SEED,
)

train_images = trainval_images[train_idx]
train_labels = trainval_labels[train_idx]
val_images   = np.concatenate([trainval_images[val_idx], val_extra_images], axis=0)
val_labels   = np.concatenate([trainval_labels[val_idx], val_extra_labels], axis=0)

print(f'Train  : {len(train_images):>5} images')
print(f'Val    : {len(val_images):>5} images  (fold1/2 val + all fold3 remainder)')
print(f'Test   : {len(test_images):>5} images  (held-out, never seen during training)')

## 6. Dataset & DataLoader

In [ ]:
# ImageNet statistics for normalisation
# ⚠ Data-leakage note: we use ImageNet stats (not computed from PanNuke train set)
#   which is standard practice for pretrained models and avoids leakage.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=90),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class PanNukeDataset(Dataset):
    """
    Lightweight wrapper — stores images as uint8 numpy arrays and
    applies transforms at __getitem__ time to keep RAM usage minimal.
    """

    def __init__(
        self,
        images: np.ndarray,   # (N, H, W, 3) uint8
        labels: np.ndarray,   # (N, 5) uint8
        transform: Optional[T.Compose] = None,
    ) -> None:
        # Store a contiguous copy so indexing is O(1)
        self.images    = np.ascontiguousarray(images, dtype=np.uint8)
        self.labels    = torch.tensor(labels, dtype=torch.float32)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        img = self.images[idx]          # (H, W, 3) uint8
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


train_ds = PanNukeDataset(train_images, train_labels, transform=train_transform)
val_ds   = PanNukeDataset(val_images,   val_labels,   transform=eval_transform)
test_ds  = PanNukeDataset(test_images,  test_labels,  transform=eval_transform)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'), drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'),
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'),
)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')

## 7. Sample Visualisation

In [ ]:
def plot_samples(
    images: np.ndarray,
    labels: np.ndarray,
    class_names: List[str],
    n: int = 8,
    title: str = 'Sample images',
) -> None:
    fig, axes = plt.subplots(2, n // 2, figsize=(18, 7))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    idx = np.random.choice(len(images), n, replace=False)
    for ax, i in zip(axes.flat, idx):
        ax.imshow(images[i])
        active = [class_names[c] for c in range(len(class_names)) if labels[i, c]]
        ax.set_title('\n'.join(active) or 'Background only', fontsize=7)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_samples(train_images, train_labels, CLASS_NAMES, title='Training set — random samples')

## 8. Model: DenseNet169 + Multi-Label Head

In [ ]:
class DenseNet169MultiLabel(nn.Module):
    """
    DenseNet169 with a two-layer classifier head for multi-label output.
    Sigmoid is NOT applied here — BCEWithLogitsLoss expects raw logits.
    """

    def __init__(self, num_classes: int, dropout_p: float = 0.3) -> None:
        super().__init__()
        backbone = models.densenet169(weights=models.DenseNet169_Weights.IMAGENET1K_V1)
        in_features = backbone.classifier.in_features   # 1664

        # Replace classifier
        backbone.classifier = nn.Identity()
        self.backbone = backbone

        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(512, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)   # (B, 1664)
        return self.classifier(features)  # (B, num_classes)


model = DenseNet169MultiLabel(num_classes=NUM_CLASSES).to(DEVICE)

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params     : {total_params:,}')
print(f'Trainable params : {trainable_params:,}')

## 9. Class Imbalance — Positive-Weight Calculation

In [ ]:
def compute_pos_weights(labels: np.ndarray) -> torch.Tensor:
    """
    pos_weight[i] = (# negative) / (# positive)  for each class.
    Passed to BCEWithLogitsLoss to up-weight rare positive classes.
    ⚠ Computed ONLY on the training set to prevent data leakage.
    """
    n_pos = labels.sum(axis=0).clip(min=1)     # (5,)
    n_neg = len(labels) - n_pos
    weights = torch.tensor(n_neg / n_pos, dtype=torch.float32)
    return weights


pos_weights = compute_pos_weights(train_labels).to(DEVICE)
print('Positive weights per class (higher = more imbalanced):')
for name, w in zip(CLASS_NAMES, pos_weights):
    print(f'  {name:<30} {w.item():.3f}')

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

## 10. Optimizer & Scheduler

In [ ]:
# Differential learning rates: smaller LR for frozen backbone, larger for head
backbone_params    = list(model.backbone.parameters())
classifier_params  = list(model.classifier.parameters())

optimizer = optim.AdamW(
    [
        {'params': backbone_params,   'lr': LR * 0.1},
        {'params': classifier_params, 'lr': LR},
    ],
    weight_decay=WEIGHT_DECAY,
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6,
)

print('Optimizer and scheduler configured.')

## 11. Training & Validation Loop

In [ ]:
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: Optional[optim.Optimizer],
    device: torch.device,
    phase: str = 'train',
) -> Tuple[float, np.ndarray, np.ndarray]:
    """
    Single epoch: returns (mean_loss, all_labels, all_probs).
    optimizer=None → eval mode (no backward pass).
    Uses torch.cuda.amp for mixed-precision to save VRAM.
    """
    is_train = (phase == 'train')
    model.train(is_train)
    scaler   = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

    total_loss  = 0.0
    all_labels  = []
    all_probs   = []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                logits = model(imgs)
                loss   = criterion(logits, lbls)

            if is_train:
                optimizer.zero_grad(set_to_none=True)   # more memory-efficient than zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * imgs.size(0)
            all_labels.append(lbls.cpu().numpy())
            all_probs.append(torch.sigmoid(logits).detach().cpu().numpy())

    all_labels = np.concatenate(all_labels, axis=0)
    all_probs  = np.concatenate(all_probs,  axis=0)
    mean_loss  = total_loss / len(loader.dataset)
    return mean_loss, all_labels, all_probs


def compute_metrics(
    labels: np.ndarray,
    probs:  np.ndarray,
    threshold: float = THRESHOLD,
) -> Dict[str, float]:
    preds = (probs >= threshold).astype(int)
    metrics = {
        'auc'      : roc_auc_score(labels, probs,   average='macro'),
        'f1_macro' : f1_score(labels, preds,         average='macro',     zero_division=0),
        'precision': precision_score(labels, preds,  average='macro',     zero_division=0),
        'recall'   : recall_score(labels, preds,     average='macro',     zero_division=0),
    }
    return metrics

In [ ]:
history: Dict[str, List[float]] = {
    'train_loss': [], 'val_loss': [],
    'train_auc':  [], 'val_auc':  [],
    'train_f1':   [], 'val_f1':   [],
}

best_val_auc   = 0.0
best_model_path = OUTPUT_DIR / 'best_densenet169_pannuke.pth'

print(f'Training for {NUM_EPOCHS} epochs on {DEVICE}...\n')

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ──
    tr_loss, tr_lbl, tr_prb = run_epoch(
        model, train_loader, criterion, optimizer, DEVICE, phase='train'
    )
    tr_metrics = compute_metrics(tr_lbl, tr_prb)

    # ── Validate ──
    vl_loss, vl_lbl, vl_prb = run_epoch(
        model, val_loader, criterion, None, DEVICE, phase='val'
    )
    vl_metrics = compute_metrics(vl_lbl, vl_prb)

    scheduler.step()

    # ── Log ──
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_auc'].append(tr_metrics['auc'])
    history['val_auc'].append(vl_metrics['auc'])
    history['train_f1'].append(tr_metrics['f1_macro'])
    history['val_f1'].append(vl_metrics['f1_macro'])

    print(
        f'Epoch {epoch:02d}/{NUM_EPOCHS} | '
        f'TrLoss {tr_loss:.4f}  TrAUC {tr_metrics["auc"]:.4f}  TrF1 {tr_metrics["f1_macro"]:.4f} | '
        f'VlLoss {vl_loss:.4f}  VlAUC {vl_metrics["auc"]:.4f}  VlF1 {vl_metrics["f1_macro"]:.4f}'
    )

    # ── Save best model (by validation AUC) ──
    if vl_metrics['auc'] > best_val_auc:
        best_val_auc = vl_metrics['auc']
        torch.save(
            {
                'epoch'      : epoch,
                'model_state': model.state_dict(),
                'optim_state': optimizer.state_dict(),
                'val_auc'    : best_val_auc,
                'class_names': CLASS_NAMES,
            },
            best_model_path,
        )
        print(f'  ✓ New best val AUC: {best_val_auc:.4f} — model saved.')

print(f'\nTraining complete. Best val AUC: {best_val_auc:.4f}')

## 12. Training Curves

In [ ]:
def plot_training_curves(history: Dict[str, List[float]], save_path: Path) -> None:
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Training History — DenseNet169 on PanNuke', fontsize=14, fontweight='bold')

    for ax, (metric, label, color_tr, color_vl) in zip(
        axes,
        [
            ('loss', 'BCE Loss',      '#2196F3', '#FF5722'),
            ('auc',  'Macro AUC',     '#4CAF50', '#9C27B0'),
            ('f1',   'Macro F1',      '#FF9800', '#607D8B'),
        ],
    ):
        ax.plot(epochs, history[f'train_{metric}'], label='Train', color=color_tr, linewidth=2)
        ax.plot(epochs, history[f'val_{metric}'],   label='Val',   color=color_vl, linewidth=2, linestyle='--')
        ax.set_title(label)
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


plot_training_curves(history, OUTPUT_DIR / 'training_curves.png')

## 13. Final Evaluation on 50 Hold-Out Test Images

In [ ]:
# Load best checkpoint
checkpoint = torch.load(best_model_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state'])
print(f"Loaded best model from epoch {checkpoint['epoch']} (val AUC: {checkpoint['val_auc']:.4f})")

_, test_labels_arr, test_probs_arr = run_epoch(
    model, test_loader, criterion, None, DEVICE, phase='val'
)
test_preds_arr = (test_probs_arr >= THRESHOLD).astype(int)

print(f'Test set size: {len(test_labels_arr)} images')

## 14. Per-Label Metrics

In [ ]:
def compute_per_label_metrics(
    labels: np.ndarray,
    probs:  np.ndarray,
    preds:  np.ndarray,
    class_names: List[str],
    threshold: float = THRESHOLD,
) -> Dict:
    results = {}
    for i, name in enumerate(class_names):
        y_true = labels[:, i]
        y_prob = probs[:, i]
        y_pred = preds[:, i]

        try:
            auc = roc_auc_score(y_true, y_prob)
        except ValueError:
            auc = float('nan')  # only one class present

        results[name] = {
            'AUC'       : round(auc, 4),
            'Accuracy'  : round(accuracy_score(y_true, y_pred), 4),
            'Precision' : round(precision_score(y_true, y_pred, zero_division=0), 4),
            'Recall'    : round(recall_score(y_true, y_pred, zero_division=0), 4),
            'F1'        : round(f1_score(y_true, y_pred, zero_division=0), 4),
        }
    return results


per_label = compute_per_label_metrics(
    test_labels_arr, test_probs_arr, test_preds_arr, CLASS_NAMES
)

# Pretty-print table
header = f"{'Class':<30} {'AUC':>6} {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6}"
print(header)
print('-' * len(header))
for cls, m in per_label.items():
    print(f"{cls:<30} {m['AUC']:>6.4f} {m['Accuracy']:>6.4f} {m['Precision']:>6.4f} {m['Recall']:>6.4f} {m['F1']:>6.4f}")

# Macro averages
macro_auc  = roc_auc_score(test_labels_arr, test_probs_arr, average='macro')
macro_f1   = f1_score(test_labels_arr, test_preds_arr, average='macro', zero_division=0)
print(f"\nMacro AUC : {macro_auc:.4f}")
print(f"Macro F1  : {macro_f1:.4f}")

# Save metrics JSON
with open(OUTPUT_DIR / 'per_label_metrics.json', 'w') as f:
    json.dump({'per_class': per_label, 'macro_auc': macro_auc, 'macro_f1': macro_f1}, f, indent=2)
print('\nMetrics saved to outputs/per_label_metrics.json')

## 15. Per-Label Metric Bar Chart

In [ ]:
def plot_per_label_metrics(
    per_label: Dict,
    class_names: List[str],
    save_path: Path,
) -> None:
    metric_keys = ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1']
    x = np.arange(len(class_names))
    width = 0.15
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

    fig, ax = plt.subplots(figsize=(14, 6))
    for i, (metric, color) in enumerate(zip(metric_keys, colors)):
        values = [per_label[cls][metric] for cls in class_names]
        bars = ax.bar(x + i * width, values, width, label=metric, color=color, alpha=0.85)
        for bar, val in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=6.5,
            )

    short_names = [n.replace('Non-neoplastic Epithelial', 'NN-Epithelial') for n in class_names]
    ax.set_xticks(x + width * (len(metric_keys) - 1) / 2)
    ax.set_xticklabels(short_names, fontsize=10)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score')
    ax.set_title('Per-Label Metrics — 50 Hold-Out Test Images', fontsize=13, fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


plot_per_label_metrics(per_label, CLASS_NAMES, OUTPUT_DIR / 'per_label_metrics.png')

## 16. Multi-Label Confusion Matrices

In [ ]:
def plot_multilabel_confusion_matrices(
    labels: np.ndarray,
    preds:  np.ndarray,
    class_names: List[str],
    save_path: Path,
) -> None:
    """
    Plot one 2×2 confusion matrix per class (one-vs-rest framing).
    """
    cms = multilabel_confusion_matrix(labels, preds)
    n   = len(class_names)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    fig.suptitle('Per-Class Confusion Matrices (One-vs-Rest)', fontsize=13, fontweight='bold')

    for ax, cm, name in zip(axes, cms, class_names):
        sns.heatmap(
            cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Neg', 'Pred Pos'],
            yticklabels=['True Neg', 'True Pos'],
            ax=ax, cbar=False,
        )
        short = name.replace('Non-neoplastic Epithelial', 'NN-Epith.')
        ax.set_title(short, fontsize=9, fontweight='bold')
        ax.tick_params(labelsize=8)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


plot_multilabel_confusion_matrices(
    test_labels_arr, test_preds_arr, CLASS_NAMES,
    OUTPUT_DIR / 'confusion_matrices.png'
)

## 17. ROC Curves Per Class

In [ ]:
from sklearn.metrics import roc_curve

def plot_roc_curves(
    labels: np.ndarray,
    probs:  np.ndarray,
    class_names: List[str],
    save_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(9, 7))
    colors = plt.cm.tab10(np.linspace(0, 1, len(class_names)))  # type: ignore

    for i, (name, color) in enumerate(zip(class_names, colors)):
        try:
            fpr, tpr, _ = roc_curve(labels[:, i], probs[:, i])
            auc_val = roc_auc_score(labels[:, i], probs[:, i])
            ax.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})', color=color, linewidth=2)
        except ValueError:
            pass  # skip if only one class in test set

    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC=0.5)')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('ROC Curves per Class — 50 Hold-Out Test Images', fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


plot_roc_curves(
    test_labels_arr, test_probs_arr, CLASS_NAMES,
    OUTPUT_DIR / 'roc_curves.png'
)

## 18. Save Test Images & Predictions

In [ ]:
def save_test_images_and_predictions(
    images: np.ndarray,
    labels: np.ndarray,
    probs:  np.ndarray,
    preds:  np.ndarray,
    class_names: List[str],
    out_dir: Path,
) -> None:
    test_img_dir = out_dir / 'test_images'
    test_img_dir.mkdir(parents=True, exist_ok=True)

    records = []
    for i in range(len(images)):
        fname = f'test_{i:03d}.png'
        img = Image.fromarray(images[i].astype(np.uint8))
        img.save(test_img_dir / fname)

        records.append({
            'filename'  : fname,
            'true_labels': {cls: int(labels[i, j]) for j, cls in enumerate(class_names)},
            'pred_probs' : {cls: round(float(probs[i, j]), 4) for j, cls in enumerate(class_names)},
            'pred_labels': {cls: int(preds[i, j]) for j, cls in enumerate(class_names)},
        })

    with open(out_dir / 'test_predictions.json', 'w') as f:
        json.dump(records, f, indent=2)

    print(f'Saved {len(images)} test images → {test_img_dir}')
    print(f'Saved predictions → {out_dir / "test_predictions.json"}')


save_test_images_and_predictions(
    test_images, test_labels_arr, test_probs_arr, test_preds_arr,
    CLASS_NAMES, OUTPUT_DIR
)

## 19. Qualitative: Predictions on Test Samples

In [ ]:
def plot_test_predictions(
    images:  np.ndarray,
    labels:  np.ndarray,
    preds:   np.ndarray,
    probs:   np.ndarray,
    class_names: List[str],
    n: int = 10,
    save_path: Optional[Path] = None,
) -> None:
    idx = np.random.choice(len(images), n, replace=False)
    fig, axes = plt.subplots(2, n // 2, figsize=(20, 8))
    fig.suptitle('Test Predictions (Green=Correct | Red=Wrong)', fontsize=13, fontweight='bold')

    for ax, i in zip(axes.flat, idx):
        ax.imshow(images[i])
        lines = []
        for j, cls in enumerate(class_names):
            gt   = int(labels[i, j])
            pred = int(preds[i, j])
            prob = probs[i, j]
            color = 'green' if gt == pred else 'red'
            short = cls[:12]
            lines.append((f'{short} {prob:.2f}', color))

        y_pos = 0.02
        for text, color in lines:
            ax.text(
                0.02, y_pos, text,
                transform=ax.transAxes,
                fontsize=6, color='white',
                bbox=dict(facecolor=color, alpha=0.7, pad=1),
            )
            y_pos += 0.13
        ax.axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


plot_test_predictions(
    test_images, test_labels_arr, test_preds_arr, test_probs_arr,
    CLASS_NAMES, n=10,
    save_path=OUTPUT_DIR / 'test_predictions_qualitative.png',
)

## 20. Output Summary

In [ ]:
print('=' * 60)
print('OUTPUT SUMMARY')
print('=' * 60)
for f in sorted(OUTPUT_DIR.rglob('*')):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f'  {str(f.relative_to(OUTPUT_DIR)):<50} {size_kb:>8.1f} KB')

print(f'\nBest Model Val AUC : {best_val_auc:.4f}')
print(f'Test Macro AUC     : {macro_auc:.4f}')
print(f'Test Macro F1      : {macro_f1:.4f}')